# Part A, Q4: Bigram Sentence ProbabilitiesUsing `Data_3.txt`, which contains three training sentences padded with `<s>` and `</s>`,plus one further padded sentence whose probability we calculate.A bigram model assumes each word depends only on the single word before it, so the probabilityof a whole sentence is the product of those pairwise conditional probabilities:$$P(w_1 \ldots w_n) = \prod_{i} P(w_i \mid w_{i-1})$$We compute this two ways.**Unsmoothed (maximum likelihood).** Straight counts:$$P(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1}, w_i)}{\text{Count}(w_{i-1})}$$**Add-one, or Laplace, smoothing.** Adds one to every bigram count so that a pair never seen intraining gets a small non-zero probability instead of collapsing the whole sentence to zero:$$P(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1}, w_i) + 1}{\text{Count}(w_{i-1}) + V}$$Probabilities are kept as exact `Fraction` objects rather than floats, so the printed outputmatches the fractions worked out by hand for the manual part of this question.**Note on V.** The vocabulary excludes `<s>`, since a sentence-start marker is never predictedas the next word, but it does include `</s>`, which can be predicted. That gives V = 9.

In [1]:
from collections import Counter
from fractions import Fraction
from pathlib import Path


START_TOKEN = "<s>"


# Walk up from the working directory to find Part_A/data, so the notebook runs
# regardless of whether the kernel starts in this folder, in Part_A, or at the repo root.
def find_data_file(filename):
    marker = Path("data") / filename
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate / marker
        if (candidate / "Part_A" / marker).exists():
            return candidate / "Part_A" / marker
    raise FileNotFoundError(f"Could not locate Part_A/data/{filename} from {Path.cwd()}")


DATA_FILE = find_data_file("Data_3.txt")
print(f"Data file: {DATA_FILE}")

Data file: C:\Users\yingx\Desktop\TextAssignment\Part_A\data\Data_3.txt


In [2]:
def load_sentences(file_path):
    # Extract only the lines that are actual padded sentences.
    with open(file_path, "r", encoding="utf-8") as file:
        sentence_lines = [
            line.strip()
            for line in file
            if line.strip().startswith(START_TOKEN)
        ]

    # The first three padded sentences are the training corpus.
    training_sentences = [line.split() for line in sentence_lines[:-1]]

    # The final padded sentence is the sentence whose probability is calculated.
    test_sentence = sentence_lines[-1].split()
    return training_sentences, test_sentence


def get_bigrams(sentence):
    # Convert a sentence into adjacent word pairs such as (<s>, I), (I, read).
    return list(zip(sentence, sentence[1:]))


def format_fraction(value):
    # Keep exact fraction output for manual probability reporting.
    return f"{value.numerator}/{value.denominator}"

## Build the countsRead the corpus, then count every individual word (unigrams) and every adjacent word pair(bigrams) across the three training sentences. These counts are the entire model.

In [3]:
training_sentences, test_sentence = load_sentences(DATA_FILE)

# Count individual words and word pairs from the training corpus.
unigram_counts = Counter()
bigram_counts = Counter()

for sentence in training_sentences:
    unigram_counts.update(sentence)
    bigram_counts.update(get_bigrams(sentence))

# <s> is not included in the predicted vocabulary for add-one smoothing.
vocabulary = sorted(token for token in unigram_counts if token != START_TOKEN)
vocabulary_size = len(vocabulary)
test_bigrams = get_bigrams(test_sentence)


print("=== Training Sentences ===")
for sentence in training_sentences:
    print(" ".join(sentence))

print("\n=== Test Sentence ===")
print(" ".join(test_sentence))

print("\n=== Vocabulary ===")
print(vocabulary)
print(f"Vocabulary size excluding <s>: {vocabulary_size}")

=== Training Sentences ===
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>

=== Test Sentence ===
<s> I read a book by Danielle </s>

=== Vocabulary ===
['</s>', 'Danielle', 'He', 'I', 'a', 'book', 'by', 'different', 'read']
Vocabulary size excluding <s>: 9


## Unsmoothed bigram modelEach conditional probability is the raw count of the pair divided by the count of the precedingword. Every bigram in the test sentence happens to appear in the training corpus, so the resultis non-zero and the zero-probability problem does not bite here. That is precisely why thesmoothed version below is worth comparing against.

In [4]:
print("\n=== Unsmoothed Bigram Model ===")
unsmoothed_probability = Fraction(1, 1)

# Unsmoothed formula:
# P(wi | wi-1) = Count(wi-1, wi) / Count(wi-1)
for previous_token, current_token in test_bigrams:
    bigram_count = bigram_counts[(previous_token, current_token)]
    previous_count = unigram_counts[previous_token]
    conditional_probability = Fraction(bigram_count, previous_count)
    unsmoothed_probability *= conditional_probability

    print(
        f"P({current_token} | {previous_token}) = "
        f"{bigram_count}/{previous_count} = {conditional_probability}"
    )

print(
    "Unsmoothed sentence probability = "
    f"{format_fraction(unsmoothed_probability)} = {float(unsmoothed_probability):.8f}"
)


=== Unsmoothed Bigram Model ===
P(I | <s>) = 1/3 = 1/3
P(read | I) = 1/1 = 1
P(a | read) = 3/3 = 1
P(book | a) = 2/3 = 2/3
P(by | book) = 1/3 = 1/3
P(Danielle | by) = 1/1 = 1
P(</s> | Danielle) = 1/1 = 1
Unsmoothed sentence probability = 2/27 = 0.07407407


## Add-one (Laplace) smoothed bigram modelAdding one to every numerator and V to every denominator redistributes a little probabilitymass to unseen pairs. The trade-off is visible in the output: the smoothed sentence probabilityis noticeably *lower* than the unsmoothed one, because mass that the unsmoothed model gaveentirely to observed pairs is now shared with pairs that were never seen.

In [5]:
print("\n=== Smoothed Bigram Model (Add-One / Laplace) ===")
smoothed_probability = Fraction(1, 1)

# Add-one smoothing formula:
# P(wi | wi-1) = (Count(wi-1, wi) + 1) / (Count(wi-1) + V)
for previous_token, current_token in test_bigrams:
    bigram_count = bigram_counts[(previous_token, current_token)]
    previous_count = unigram_counts[previous_token]
    numerator = bigram_count + 1
    denominator = previous_count + vocabulary_size
    conditional_probability = Fraction(numerator, denominator)
    smoothed_probability *= conditional_probability

    print(
        f"P({current_token} | {previous_token}) = "
        f"({bigram_count}+1)/({previous_count}+{vocabulary_size}) = "
        f"{numerator}/{denominator} = {conditional_probability}"
    )

print(
    "Smoothed sentence probability = "
    f"{format_fraction(smoothed_probability)} = {float(smoothed_probability):.8f}"
)


=== Smoothed Bigram Model (Add-One / Laplace) ===
P(I | <s>) = (1+1)/(3+9) = 2/12 = 1/6
P(read | I) = (1+1)/(1+9) = 2/10 = 1/5
P(a | read) = (3+1)/(3+9) = 4/12 = 1/3
P(book | a) = (2+1)/(3+9) = 3/12 = 1/4
P(by | book) = (1+1)/(3+9) = 2/12 = 1/6
P(Danielle | by) = (1+1)/(1+9) = 2/10 = 1/5
P(</s> | Danielle) = (1+1)/(1+9) = 2/10 = 1/5
Smoothed sentence probability = 1/54000 = 0.00001852
